# Task 1 — Data Exploration and Enrichment

This notebook covers Task 1.1 and 1.2: understanding the schema of the Ethiopia
Financial Inclusion unified dataset, and exploring its contents (counts by
record_type/pillar/source_type/confidence, temporal range, indicator coverage,
event catalog, and impact_link relationships) before enrichment.

In [12]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
import matplotlib.pyplot as plt

from data_loader import load_all, valid_codes

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

data, links, ref = load_all()
print(f"Main data sheet: {data.shape[0]} records, {data.shape[1]} columns")
print(f"Impact_links sheet: {links.shape[0]} records, {links.shape[1]} columns")
print(f"Reference codes: {ref.shape[0]} rows")

Main data sheet: 43 records, 34 columns
Impact_links sheet: 14 records, 35 columns
Reference codes: 71 rows


## 1.1 Understand the Schema

In [31]:
# The main sheet mixes three record_types in one table: observation, event, target.
# Every record shares the same 34 columns; which columns are populated depends on record_type.
data[["record_id", "record_type", "category", "pillar", "indicator", "indicator_code",
      "value_numeric", "observation_date"]].head(10)

,record_id,record_type,category,pillar,indicator,indicator_code,value_numeric,observation_date
0,REC_0001,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,22.00,2014-12-31
1,REC_0002,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,35.00,2017-12-31
2,REC_0003,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,46.00,2021-12-31
3,REC_0004,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,56.00,2021-12-31
4,REC_0005,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,36.00,2021-12-31
5,REC_0006,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,49.00,2024-11-29
6,REC_0007,observation,NaN,ACCESS,Mobile Money Account Rate,ACC_MM_ACCOUNT,4.70,2021-12-31
7,REC_0008,observation,NaN,ACCESS,Mobile Money Account Rate,ACC_MM_ACCOUNT,9.45,2024-11-29
8,REC_0009,observation,NaN,ACCESS,4G Population Coverage,ACC_4G_COV,37.50,2023-06-30
9,REC_0010,observation,NaN,ACCESS,4G Population Coverage,ACC_4G_COV,70.80,2025-06-30


In [32]:
data[data["record_type"] == "event"][
    ["record_id", "record_type", "category", "pillar", "indicator", "observation_date"]
].head(10)

,record_id,record_type,category,pillar,indicator,observation_date
33,EVT_0001,event,product_launch,NaN,Telebirr Launch,2021-05-17
34,EVT_0002,event,market_entry,NaN,Safaricom Ethiopia Commercial Launch,2022-08-01
35,EVT_0003,event,product_launch,NaN,M-Pesa Ethiopia Launch,2023-08-01
36,EVT_0004,event,infrastructure,NaN,Fayda Digital ID Program Rollout,2024-01-01
37,EVT_0005,event,policy,NaN,Foreign Exchange Liberalization,2024-07-29
38,EVT_0006,event,milestone,NaN,P2P Transaction Count Surpasses ATM,2024-10-01
39,EVT_0007,event,partnership,NaN,M-Pesa EthSwitch Integration,2025-10-27
40,EVT_0008,event,infrastructure,NaN,EthioPay Instant Payment System Launch,2025-12-18
41,EVT_0009,event,policy,NaN,NFIS-II Strategy Launch,2021-09-01
42,EVT_0010,event,pricing,NaN,Safaricom Ethiopia Price Increase,2025-12-15


In [3]:
# impact_links connect an event to an indicator via parent_id, which references
# an event's record_id in the main sheet (e.g. EVT_0001 = Telebirr Launch).
links[["record_id", "parent_id", "related_indicator", "relationship_type",
       "impact_direction", "impact_magnitude", "lag_months", "evidence_basis"]].head(10)

,record_id,parent_id,related_indicator,relationship_type,impact_direction,impact_magnitude,lag_months,evidence_basis
0,IMP_0001,EVT_0001,ACC_OWNERSHIP,direct,increase,high,12,literature
1,IMP_0002,EVT_0001,USG_TELEBIRR_USERS,direct,increase,high,3,empirical
2,IMP_0003,EVT_0001,USG_P2P_COUNT,direct,increase,high,6,empirical
3,IMP_0004,EVT_0002,ACC_4G_COV,direct,increase,medium,12,empirical
4,IMP_0005,EVT_0002,AFF_DATA_INCOME,indirect,decrease,medium,12,literature
5,IMP_0006,EVT_0003,USG_MPESA_USERS,direct,increase,high,3,empirical
6,IMP_0007,EVT_0003,ACC_MM_ACCOUNT,direct,increase,medium,6,theoretical
7,IMP_0008,EVT_0004,ACC_OWNERSHIP,enabling,increase,medium,24,literature
8,IMP_0009,EVT_0004,GEN_GAP_ACC,indirect,decrease,medium,24,literature
9,IMP_0010,EVT_0005,AFF_DATA_INCOME,indirect,increase,high,3,empirical


In [29]:
# Confirm every parent_id in impact_links actually resolves to an event in the main sheet.
event_ids = set(data.loc[data["record_type"] == "event", "record_id"])
orphaned = links[~links["parent_id"].isin(event_ids)]
print(f"impact_link records with a valid parent event: {len(links) - len(orphaned)} / {len(links)}")
orphaned

impact_link records with a valid parent event: 14 / 14


,record_id,parent_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,unit,observation_date,period_start,period_end,fiscal_year,gender,location,region,source_name,source_type,source_url,confidence,related_indicator,relationship_type,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes


**Schema notes:**
- The main sheet holds three `record_type`s: `observation` (actual measured values),
  `event` (policies, product launches, milestones), and `target` (official policy goals).
- `category` (e.g. `policy`, `product_launch`, `infrastructure`) is populated **only**
  for events — observations and targets leave it blank, since a category describing
  the *type of event* doesn't apply to a measured value.
- `pillar` (e.g. `ACCESS`, `USAGE`, `GENDER`) is populated for observations and targets,
  but **left blank for events** in the starter data. This is a deliberate design choice
  worth noting as a modeling challenge (see below), not a data quality bug.
- Every `impact_link.parent_id` in the starter data resolves to a real event's
  `record_id` — the linkage is clean.

**The pillar-for-events challenge:** unlike observations, a single event often
plausibly affects *multiple* pillars at once — e.g. the Telebirr launch (a
`product_launch` event) has impact_links touching both `ACCESS`
(`ACC_OWNERSHIP`) and `USAGE` (`USG_TELEBIRR_USERS`, `USG_P2P_COUNT`). Assigning
one fixed `pillar` value to the event record itself would understate this —
which is exactly why the schema instead captures pillar-level effects
per-relationship, in the `impact_links` sheet, rather than on the event record.
When adding new events, we follow the same convention: leave `pillar` blank on
the event, and encode each affected pillar as a separate `impact_link` row.

## 1.2 Explore the Data

In [28]:
for col in ["record_type", "pillar", "source_type", "confidence"]:
    print(f"--- {col} ---")
    print(data[col].value_counts(dropna=False))
    print()


--- record_type ---
record_type
observation    30
event          10
target          3
Name: count, dtype: int64

--- pillar ---
pillar
ACCESS           16
USAGE            11
NaN              10
GENDER            5
AFFORDABILITY     1
Name: count, dtype: int64

--- source_type ---
source_type
operator      15
survey        10
regulator      7
research       4
policy         3
calculated     2
news           2
Name: count, dtype: int64

--- confidence ---
confidence
high      40
medium     3
Name: count, dtype: int64



In [27]:
obs = data[data["record_type"] == "observation"]
print("Temporal range of observations:")
print("Earliest:", obs["observation_date"].min().date())
print("Latest:  ", obs["observation_date"].max().date())
print(f"Span: {(obs['observation_date'].max() - obs['observation_date'].min()).days / 365.25:.1f} years")



Temporal range of observations:
Earliest: 2014-12-31
Latest:   2025-12-31
Span: 11.0 years


In [22]:
indicator_coverage = (
    obs.groupby(["indicator_code", "indicator", "pillar"])
    .agg(n_observations=("value_numeric", "count"),
         first_date=("observation_date", "min"),
         last_date=("observation_date", "max"))
    .reset_index()
    .sort_values("pillar")
)
indicator_coverage

,indicator_code,indicator,pillar,n_observations,first_date,last_date
0,ACC_4G_COV,4G Population Coverage,ACCESS,2,2023-06-30,2025-06-30
1,ACC_FAYDA,Fayda Digital ID Enrollment,ACCESS,3,2024-08-15,2025-05-15
2,ACC_MM_ACCOUNT,Mobile Money Account Rate,ACCESS,2,2021-12-31,2024-11-29
3,ACC_MOBILE_PEN,Mobile Subscription Penetration,ACCESS,1,2025-12-31,2025-12-31
4,ACC_OWNERSHIP,Account Ownership Rate,ACCESS,6,2014-12-31,2024-11-29
5,AFF_DATA_INCOME,Data Affordability Index,AFFORDABILITY,1,2024-12-31,2024-12-31
6,GEN_GAP_ACC,Account Ownership Gender Gap,GENDER,2,2021-12-31,2024-11-29
7,GEN_GAP_MOBILE,Mobile Phone Gender Gap,GENDER,1,2024-12-31,2024-12-31
8,GEN_MM_SHARE,Female Mobile Money Account Share,GENDER,1,2024-12-31,2024-12-31
16,USG_P2P_VALUE,P2P Transaction Value,USAGE,1,2025-07-07,2025-07-07


**Coverage observation:** most indicators have only 1-3 data points, and several
(`ACC_MOBILE_PEN`, `USG_P2P_VALUE`, `USG_ATM_COUNT`, `USG_ATM_VALUE`,
`USG_CROSSOVER`, `AFF_DATA_INCOME`, and all `GENDER` indicators except
`GEN_GAP_ACC`) have exactly **one** observation — not enough to establish a
trend on their own. This is the central gap Task 1's enrichment should address:
more historical depth on the indicators that matter most for forecasting
Access and Usage.

In [23]:
events = data[data["record_type"] == "event"].sort_values("observation_date")
events[["record_id", "observation_date", "category", "indicator", "source_type", "confidence"]]

,record_id,observation_date,category,indicator,source_type,confidence
33,EVT_0001,2021-05-17,product_launch,Telebirr Launch,operator,high
41,EVT_0009,2021-09-01,policy,NFIS-II Strategy Launch,regulator,high
34,EVT_0002,2022-08-01,market_entry,Safaricom Ethiopia Commercial Launch,news,high
35,EVT_0003,2023-08-01,product_launch,M-Pesa Ethiopia Launch,operator,high
36,EVT_0004,2024-01-01,infrastructure,Fayda Digital ID Program Rollout,regulator,high
37,EVT_0005,2024-07-29,policy,Foreign Exchange Liberalization,regulator,high
38,EVT_0006,2024-10-01,milestone,P2P Transaction Count Surpasses ATM,operator,high
39,EVT_0007,2025-10-27,partnership,M-Pesa EthSwitch Integration,operator,high
42,EVT_0010,2025-12-15,pricing,Safaricom Ethiopia Price Increase,news,high
40,EVT_0008,2025-12-18,infrastructure,EthioPay Instant Payment System Launch,regulator,high


In [10]:
targets = data[data["record_type"] == "target"]
targets[["record_id", "pillar", "indicator", "indicator_code", "value_numeric", "observation_date"]]

,record_id,pillar,indicator,indicator_code,value_numeric,observation_date
30,REC_0031,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,70.0,2025-12-31
31,REC_0032,ACCESS,Fayda Digital ID Enrollment,ACC_FAYDA,90000000.0,2028-12-31
32,REC_0033,GENDER,Female Mobile Money Account Share,GEN_MM_SHARE,50.0,2030-12-31


In [26]:
print("impact_link relationship_type counts:")
print(links["relationship_type"].value_counts())
print()
print("impact_link evidence_basis counts:")
print(links["evidence_basis"].value_counts())
print()
print("Events with at least one impact_link vs. events with none:")
linked_events = set(links["parent_id"])
print(f"Linked: {len(linked_events)} / {len(event_ids)}")
print("Not yet linked to any indicator:", event_ids - linked_events)


impact_link relationship_type counts:
relationship_type
direct      9
indirect    4
enabling    1
Name: count, dtype: int64

impact_link evidence_basis counts:
evidence_basis
literature     7
empirical      6
theoretical    1
Name: count, dtype: int64

Events with at least one impact_link vs. events with none:
Linked: 8 / 10
Not yet linked to any indicator: {'EVT_0009', 'EVT_0006'}


**Coverage summary:**
- 30 observations, 10 events, 3 targets, and 14 impact_links in the starter data.
- Observations span **2014-12-31 to 2025-12-31** (~11 years), though most indicators
  individually only have 1-3 data points within that span.
- All 10 events have at least one `impact_link`, mostly `direct` relationships based
  on `literature` (cross-country comparisons) or `empirical` (observed Ethiopian data)
  evidence — a mix worth being explicit about when we add our own impact_links below.

This sets up Task 1.3: enriching the dataset with additional observations (more
history per indicator, disaggregations), events (policy/infrastructure milestones
not yet captured), and impact_links (plausible new event-to-indicator relationships),
each with a real, citable source.